# Anomaly Detection EDA

This notebook works in both Google Colab and local Jupyter.

- Colab uses the Google Drive parquet path
- Local mode uses `../dataset` and falls back to `./dataset`
- Iteration 1 focuses on schema validation and basic sanity checks


In [19]:
from pathlib import Path
import importlib.util
import subprocess
import sys


def running_in_colab():
    return importlib.util.find_spec("google.colab") is not None


def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or import_name])
    return __import__(import_name)


IS_COLAB = running_in_colab()
duckdb = ensure_package("duckdb")
pd = ensure_package("pandas")
con = duckdb.connect()


def query_df(query):
    return con.sql(query).df()


print(f"Running in Colab: {IS_COLAB}")
print(f"Python executable: {sys.executable}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"Pandas version: {pd.__version__}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 55.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 55.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Running in Colab: False
Python executable: /Users/harrish/Desktop/practicum/.venv/bin/python
DuckDB version: 1.5.2
Pandas version: 3.0.3


In [20]:
if IS_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    dataset_root = Path("/content/drive/MyDrive/Practicum/NGIDS/NGIDS-DS-v1/parquet")
else:
    dataset_root = Path("../dataset")
    if not dataset_root.exists():
        dataset_root = Path("dataset")

host_logs = dataset_root / "host_logs.parquet"
ground_truth = dataset_root / "ground_truth.parquet"

print(f"dataset_root: {dataset_root}")
print(f"host_logs parquet: {host_logs}")
print(f"ground_truth parquet: {ground_truth}")


dataset_root: ../dataset
host_logs parquet: ../dataset/host_logs.parquet
ground_truth parquet: ../dataset/ground_truth.parquet


## Iteration 1: Dataset Orientation

This section verifies row counts, schemas, sample rows, time ranges, and a few label-like columns.


In [21]:
query_df(f"""
SELECT 'host_logs' AS dataset, count(*) AS row_count FROM '{host_logs}'
UNION ALL
SELECT 'ground_truth' AS dataset, count(*) AS row_count FROM '{ground_truth}'
""")


,dataset,row_count
0,host_logs,90054239
1,ground_truth,313926


In [22]:
query_df(f"DESCRIBE SELECT * FROM '{host_logs}'")


,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,TIME,YES,None,None,None
2,pro_id,BIGINT,YES,None,None,None
3,path,VARCHAR,YES,None,None,None
4,sys_call,BIGINT,YES,None,None,None
5,event_id,BIGINT,YES,None,None,None
6,attack_cat,VARCHAR,YES,None,None,None
7,attack_subcat,VARCHAR,YES,None,None,None
8,label,BIGINT,YES,None,None,None


In [31]:
query_df(f"SELECT * FROM '{host_logs}' LIMIT 10")


,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label
0,2016-03-11,02:45:01,1830,/sbin/upstart-dbus-bridge,142,45354,normal,normal,0
1,2016-03-11,02:45:06,1804,/bin/dbus-daemon,256,45352,normal,normal,0
2,2016-03-11,02:45:06,2133,/usr/lib/i386-linux-gnu/gconf/gconfd-2,168,45372,normal,normal,0
3,2016-03-11,02:45:35,4528,/usr/bin/python3.4,3,39459,normal,normal,0
4,2016-03-11,02:45:44,1847,/usr/bin/ibus-daemon,102,37263,normal,normal,0
5,2016-03-11,02:45:44,1907,/usr/lib/ibus/ibus-ui-gtk3,168,37896,normal,normal,0
6,2016-03-11,02:45:44,1925,/usr/lib/ibus/ibus-engine-simple,168,37542,normal,normal,0
7,2016-03-11,02:45:44,4461,/usr/sbin/apache2,142,37647,normal,normal,0
8,2016-03-11,02:45:45,1081,/usr/bin/Xorg,102,37480,normal,normal,0
9,2016-03-11,02:45:11,3989,/sbin/auditd,256,45374,normal,normal,0


In [ ]:
query_df(f"""
SELECT
    min(date) AS min_date,
    max(date) AS max_date,
    min(time) AS min_time,
    max(time) AS max_time,
    count(*) AS row_count,
    count(DISTINCT row(date, time, pro_id, path, sys_call, event_id, attack_cat, attack_subcat, label)) AS distinct_rows,
    count(*) - count(DISTINCT row(date, time, pro_id, path, sys_call, event_id, attack_cat, attack_subcat, label)) AS duplicate_rows
FROM '{host_logs}'
""")


In [ ]:
query_df(f"""
SELECT attack_cat, attack_subcat, label, count(*) AS n
FROM '{host_logs}'
GROUP BY 1, 2, 3
ORDER BY n DESC
LIMIT 20
""")


In [1]:
query_df(f"""
SELECT
    count(DISTINCT pro_id) AS distinct_pro_id,
    count(DISTINCT path) AS distinct_path,
    count(DISTINCT sys_call) AS distinct_sys_call,
    count(DISTINCT event_id) AS distinct_event_id,
    count(DISTINCT attack_cat) AS distinct_attack_cat,
    count(DISTINCT attack_subcat) AS distinct_attack_subcat,
    count(DISTINCT label) AS distinct_label
FROM '{host_logs}'
""")


NameError: name 'query_df' is not defined

In [ ]:
query_df(f"DESCRIBE SELECT * FROM '{ground_truth}'")


In [ ]:
query_df(f"SELECT * FROM '{ground_truth}' LIMIT 10")


In [ ]:
query_df(f"""
SELECT
    min(date) AS min_date,
    max(date) AS max_date,
    count(*) AS row_count,
    count(DISTINCT row(date, time, attack_cat, attack_subcat, attack_name, attack_refrence, ips)) AS distinct_rows,
    count(*) - count(DISTINCT row(date, time, attack_cat, attack_subcat, attack_name, attack_refrence, ips)) AS duplicate_rows,
    count(*) FILTER (WHERE time = 'Time') AS header_like_rows
FROM '{ground_truth}'
""")


In [ ]:
query_df(f"""
SELECT attack_cat, count(*) AS n
FROM '{ground_truth}'
GROUP BY 1
ORDER BY n DESC
LIMIT 20
""")


In [ ]:
query_df(f"SELECT * FROM '{ground_truth}' WHERE time = 'Time' LIMIT 10")
